# 🚀 Google Colab GPU Backend - Tạo Mô Hình 3D Từ Ảnh (TripoSR)
### 100% Miễn Phí • Không Cần Mua API Key • Xử Lý Cực Nhanh (~1.5s trên GPU T4)

**Hướng dẫn 3 bước cực nhanh:**
1. Vào menu **Runtime** > **Change runtime type** > Chọn **T4 GPU** (nếu chưa chọn).
2. Nhấn nút **Run all** (hoặc chạy lần lượt 2 ô bên dưới).
3. Sao chép link **Cloudflare Tunnel** (`https://...trycloudflare.com`) xuất hiện ở ô số 2, sau đó dán vào mục **Cài đặt Engine** trên trang web của bạn!


In [ ]:
# BƯỚC 1: Cài đặt thư viện AI, torchmcubes, PyMCubes & TripoSR (Chạy mất khoảng 1-2 phút)
!nvidia-smi
!pip install -q ninja cmake pybind11 scikit-build-core PyMCubes
!pip install -q "rembg[gpu]" onnxruntime-gpu einops omegaconf transformers trimesh
!pip install -q --no-build-isolation git+https://github.com/tatsy/torchmcubes.git
!pip install -q fastapi uvicorn python-multipart pyvista xatlas gradio pyngrok
# Đảm bảo Pillow tương thích hoàn toàn với Python 3.13 trên Colab
!pip install -q --upgrade --force-reinstall pillow

# Tải mã nguồn TripoSR nếu chưa có
![ ! -d "/content/TripoSR" ] && git clone https://github.com/VAST-AI-Research/TripoSR.git /content/TripoSR

# Tải cloudflared để tạo đường hầm HTTPS miễn phí vĩnh viễn không cần đăng ký
![ ! -f "/usr/local/bin/cloudflared" ] && wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

print('\n✅ Cài đặt môi trường GPU & TripoSR hoàn tất!')


In [ ]:
# BƯỚC 2: Khởi động API Server TripoSR & Mở Link Public
import os, sys, io, time, re, subprocess, threading, types

# Tự động kiểm tra và xử lý torchmcubes
try:
    import torchmcubes
    print('✅ Đã nạp module torchmcubes!')
except ImportError:
    print('⚡ Đang cài đặt torchmcubes với --no-build-isolation...')
    try:
        subprocess.check_call(['pip', 'install', '-q', '--no-build-isolation', 'git+https://github.com/tatsy/torchmcubes.git'])
        import torchmcubes
        print('✅ Cài đặt torchmcubes thành công!')
    except Exception as e:
        print(f'⚠️ Không biên dịch được torchmcubes ({e}). Kích hoạt PyMCubes fallback...')
        import mcubes
        import torch
        def custom_mc(density, isovalue=0.0):
            d_np = density.detach().cpu().numpy()
            verts, tris = mcubes.marching_cubes(d_np, isovalue)
            return torch.from_numpy(verts).float(), torch.from_numpy(tris.astype('int64'))
        fake_module = types.ModuleType('torchmcubes')
        fake_module.marching_cubes = custom_mc
        sys.modules['torchmcubes'] = fake_module
        print('✅ PyMCubes fallback đã sẵn sàng!')

# Tự động kiểm tra & sửa lỗi Pillow nếu bị xung đột trên Python 3.13
try:
    from PIL import Image
    import PIL._typing
except Exception:
    print('⚡ Đang khắc phục lỗi tương thích Pillow...')
    subprocess.check_call(['pip', 'install', '-q', '--upgrade', '--force-reinstall', 'pillow'])
    import importlib, PIL
    importlib.reload(PIL)
    from PIL import Image

import torch
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse, Response, JSONResponse
import uvicorn

sys.path.append('/content/TripoSR')
from tsr.system import TSR
from tsr.utils import remove_background, resize_foreground, to_gradio_3d_orientation

# Khởi tạo rembg tách phông nền an toàn
rembg_session = None
try:
    import rembg
    rembg_session = rembg.new_session()
    print('✅ Rembg AI (tách nền ảnh) đã sẵn sàng!')
except Exception as e:
    print(f'⚠️ Cảnh báo rembg ({e}), sẽ dùng ảnh gốc.')

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f'⚡ Đang nạp mô hình TripoSR vào {device}...')
model = TSR.from_pretrained('stabilityai/TripoSR', config_name='config.yaml', weight_name='model.ckpt')
model.renderer.set_chunk_size(8192)
model.to(device)
print('✅ Mô hình TripoSR đã sẵn sàng trên GPU!')

app = FastAPI(title='TripoSR 3D Backend')
app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

os.makedirs('/content/output', exist_ok=True)

@app.get('/')
@app.get('/health')
def health():
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
    return {'status': 'ok', 'engine': 'TripoSR', 'device': device, 'gpu': gpu_name}

@app.post('/generate')
async def generate_3d(image: UploadFile = File(...), do_remove_bg: bool = Form(True), foreground_ratio: float = Form(0.85), mc_resolution: int = Form(256)):
    try:
        start_time = time.time()
        image_bytes = await image.read()
        input_image = Image.open(io.BytesIO(image_bytes))
        if input_image.mode != 'RGB':
            input_image = input_image.convert('RGB')

        if do_remove_bg and rembg_session is not None:
            try:
                rembg_img = remove_background(input_image, rembg_session)
                input_image = resize_foreground(rembg_img, foreground_ratio)
            except Exception as bg_err:
                print(f'Lỗi tách nền: {bg_err}')

        with torch.no_grad():
            scene_codes = model([input_image], device=device)
            mesh = model.extract_mesh(scene_codes, resolution=mc_resolution)[0]
            mesh = to_gradio_3d_orientation(mesh)

        out_filename = f'model_{int(time.time()*1000)}.glb'
        out_path = os.path.join('/content/output', out_filename)
        mesh.export(out_path)
        elapsed = round(time.time() - start_time, 2)
        print(f'✨ Đã tạo xong 3D trong {elapsed} giây: {out_filename}')
        return FileResponse(out_path, media_type='model/gltf-binary', filename=out_filename)
    except Exception as e:
        print(f'❌ Lỗi tạo 3D: {e}')
        return JSONResponse(status_code=500, content={'error': str(e)})

# Khởi chạy server uvicorn trong thread ngầm
def run_server():
    uvicorn.run(app, host='127.0.0.1', port=8000, log_level='warning')

threading.Thread(target=run_server, daemon=True).start()
time.sleep(2)

# Chạy Cloudflare Tunnel để tạo URL HTTPS công khai
print('\n🌐 Đang tạo đường hầm Cloudflare Tunnel...')
subprocess.Popen('cloudflared tunnel --url http://127.0.0.1:8000 > /tmp/cf.log 2>&1', shell=True)

tunnel_url = None
for _ in range(25):
    time.sleep(1)
    if os.path.exists('/tmp/cf.log'):
        with open('/tmp/cf.log', 'r') as f:
            text = f.read()
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', text)
            if match:
                tunnel_url = match.group(0)
                break

print('='*65)
if tunnel_url:
    print('🎉 BACKEND COLAB ĐÃ SẴN SÀNG!')
    print(f'👉 COPY LINK DƯỚI ĐÂY DÁN VÀO WEB CỦA BẠN:\n\n{tunnel_url}\n')
else:
    print('⚠️ Đang tạo link tunnel... Nếu chưa hiện, hãy chạy lại ô này sau 5 giây.')
print('='*65)
